# Нейроюрист по сделкам с жилой недвижимостью — прототип (Этап № 3)

**Тема.** Нейро-сотрудник — юрист-консультант для физических лиц по сделкам с **жилой** недвижимостью:
купля-продажа, наём, ипотека, участие в долевом строительстве (ДДУ), дарение, рента и пожизненное
содержание с иждивением, наследование жилья, налоги при продаже/дарении/наследовании, государственная
регистрация прав.

**Задача.** По бытовому вопросу пользователя дать корректную консультацию **со ссылками на конкретные
нормы** (статьи ГК/ЖК/СК/НК, федеральные законы, постановления Пленума ВС РФ, судебную практику),
не выходя за периметр жилой недвижимости и не придумывая нормы, которых нет в базе.

**Алгоритм (RAG + многошаговая цепочка LLM).**
1. Разбор вопроса и сборка поисковых запросов (вызов LLM № 1).
2. Поиск фрагментов в векторной базе FAISS + генерация ответа со ссылками (вызов LLM № 2).
3. Проверка ответа на соответствие найденным источникам, при необходимости — один повтор генерации (вызов LLM № 3).

Плюс — ведение истории диалога (уточняющие вопросы).


## 1. Установка зависимостей

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu openai tiktoken

## 2. База знаний и её обработка

База знаний собрана вручную из открытых официальных источников (ГАРАНT, КонсультантПлюс, портал ВС РФ,
справочник постатейной практики gkrfkod.ru) и хранится в отдельном GitHub-репозитории — тексты
законов и судебных актов скачаны/скопированы из первоисточника и извлечены локально (`pypdf` / `python-docx`),
**без автоматического парсинга сайтов и без прогона через языковую модель**: для юридического RAG точность
цитирования нормы критична.

Состав (~5.4 МБ, 87 текстовых файлов):

| Каталог | Файлов | Что внутри |
|---|---|---|
| `laws/` | 25 | ГК РФ (гл. 30, 32–35, 61–64 + точечные статьи), ЖК РФ (гл. 5–6), СК РФ (гл. 7–8), ФЗ-218, ФЗ-102, ФЗ-214, НК РФ (ст. 217/217.1/220) |
| `plenum/` | 10 | Постановления Пленума ВС РФ по темам периметра |
| `practice/` | 48 | Судебная практика по конкретным статьям ГК (до 5 актов ВС РФ на статью) |
| `obzory/` | 4 | Точечные пункты обзоров практики Президиума ВС РФ |

Конфиденциальных / персональных данных в базе нет — только опубликованные нормативные акты и
обезличенная судебная практика ВС РФ.

Обработка в ноутбуке: загрузка текстов → разбиение на чанки с сохранением метаданных (файл-источник,
раздел базы) → построение эмбеддингов (`text-embedding-3-small`) → индекс FAISS.


In [ ]:
# Клонируем репозиторий с базой знаний.
REPO_URL = "https://github.com/vaar970-cmyk/neurojurist-realestate.git"
REPO_DIR = "neurojurist-realestate"

import os, shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 {REPO_URL} {REPO_DIR}

KB_DIR = os.path.join(REPO_DIR, "knowledge-base")
TESTSET_PATH = os.path.join(REPO_DIR, "testset", "test_questions.md")
print("Каталоги базы знаний:", sorted(os.listdir(KB_DIR)))

### 2.1. Ключ OpenAI

Ключ берётся из «секретов» Colab (значок 🔑 слева, имя `OPENAI_API_KEY`).
Если секрет не задан — ноутбук спросит ключ через безопасный ввод.


In [ ]:
import os, getpass

def setup_api_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
    if not key:
        key = getpass.getpass("Введите OPENAI_API_KEY: ")
    os.environ["OPENAI_API_KEY"] = key

setup_api_key()
print("Ключ установлен:", bool(os.environ.get("OPENAI_API_KEY")))

### 2.2. Загрузка текстов базы знаний

Каждый файл превращается в `Document` с метаданными:
* `source` — имя файла (для ссылки в ответе видно, из какого документа фрагмент);
* `category` — раздел базы (`laws` / `plenum` / `practice` / `obzory`).


In [ ]:
import glob
from langchain_core.documents import Document

CATEGORY_TITLES = {
    "laws": "законодательство",
    "plenum": "постановление Пленума ВС РФ",
    "practice": "судебная практика по статье",
    "obzory": "обзор практики ВС РФ",
}

def load_knowledge_base(kb_dir):
    docs = []
    for category in CATEGORY_TITLES:
        for path in sorted(glob.glob(os.path.join(kb_dir, category, "*.txt"))):
            text = open(path, encoding="utf-8").read().strip()
            if not text:
                continue
            docs.append(Document(
                page_content=text,
                metadata={"source": os.path.basename(path), "category": category},
            ))
    return docs

raw_docs = load_knowledge_base(KB_DIR)

from collections import Counter
by_cat = Counter(d.metadata["category"] for d in raw_docs)
total_chars = sum(len(d.page_content) for d in raw_docs)
print(f"Загружено документов: {len(raw_docs)}")
for cat, n in by_cat.items():
    print(f"  {cat:9s}: {n} файлов")
print(f"Суммарный объём текста: {total_chars/1_000_000:.2f} млн символов")

### 2.3. Разбиение на чанки

Используем `RecursiveCharacterTextSplitter`. Список разделителей начинается с `«\nСтатья »` и
`«\nПостановление »` — так сплиттер старается резать текст по границам статей и судебных актов,
а не посреди нормы. Метаданные исходного документа наследуются каждым чанком.

`chunk_size` и `chunk_overlap` — гиперпараметры; их влияние на качество поиска будет сравниваться
в экспериментальной части (Этап № 4). Здесь взяты рабочие значения по умолчанию.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\nСтатья ", "\nПостановление ", "\nОпределение ", "\n\n", "\n", ". ", " ", ""],
    keep_separator=True,
)

chunks = splitter.split_documents(raw_docs)
print(f"Чанков всего: {len(chunks)}")
print(f"Средняя длина чанка: {sum(len(c.page_content) for c in chunks)//len(chunks)} символов")
by_cat_chunks = Counter(c.metadata["category"] for c in chunks)
for cat, n in by_cat_chunks.items():
    print(f"  {cat:9s}: {n} чанков")
print("\nПример чанка:\n" + "-"*60)
print(chunks[0].page_content[:400], "...")
print("метаданные:", chunks[0].metadata)

### 2.4. Векторный индекс FAISS

Эмбеддинги — `text-embedding-3-small` (OpenAI). Индекс строится один раз и сохраняется локально,
чтобы при повторном запуске нижних ячеек не пересчитывать эмбеддинги.


In [ ]:
import time
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

INDEX_DIR = "faiss_index"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

if os.path.isdir(INDEX_DIR):
    knowledge_base = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
    print("Индекс загружен с диска.")
else:
    t0 = time.time()
    knowledge_base = FAISS.from_documents(chunks, embeddings)
    knowledge_base.save_local(INDEX_DIR)
    print(f"Индекс построен за {time.time()-t0:.1f} с и сохранён в {INDEX_DIR}/")

print("Векторов в индексе:", knowledge_base.index.ntotal)

### 2.5. Проверка поиска (sanity-check)

Быстрый тест: по бытовому запросу индекс должен возвращать релевантные фрагменты нужных статей.


In [ ]:
for q in ["переход права собственности на квартиру после подписания договора",
          "согласие супруга на продажу квартиры"]:
    print(f"\nЗапрос: {q}")
    for d in knowledge_base.similarity_search(q, k=3):
        head = d.page_content.strip().split("\n", 1)[0][:80]
        print(f"  [{d.metadata['source']}] {head}")

## 3. Структура нейро-сотрудника

```
                 ┌───────────────────────────────────────────────────────────┐
   вопрос  ─────▶ │  ШАГ 1. Разбор вопроса и сборка поисковых запросов (LLM)  │
 (+ история,      │  бытовой вопрос → юридические подвопросы + точные запросы  │
  + тема меню)    │  выход: JSON {themes: [...], queries: [...]}               │
                 └───────────────────────────┬───────────────────────────────┘
                                             │ queries[]
                                             ▼
                 ┌───────────────────────────────────────────────────────────┐
                 │  ШАГ 2. Поиск в FAISS + генерация ответа (LLM)            │
                 │  по каждому запросу similarity_search(k) → объединение    │
                 │  уникальных чанков → ответ со ссылками на нормы           │
                 └───────────────────────────┬───────────────────────────────┘
                                             │ черновик ответа + контекст
                                             ▼
                 ┌───────────────────────────────────────────────────────────┐
                 │  ШАГ 3. Проверка ответа (LLM)                            │
                 │  сверка каждой ссылки и тезиса с найденными фрагментами   │
                 │  JSON {ok: bool, problems: [...]}                         │
                 │  если ok=false → один повтор ШАГА 2 с учётом замечаний    │
                 └───────────────────────────┬───────────────────────────────┘
                                             │ финальный ответ
                                             ▼
                                        пользователь
                                             │
                          история диалога ◀──┘  (вопрос + ответ добавляются в буфер,
                                                 используются на ШАГЕ 1 следующего вопроса)
```

**Блоки:**
* **Шаг 1 — разбор и сборка запроса.** Один вызов LLM. Переводит бытовую формулировку в юридические
  категории и разбивает составной вопрос на подвопросы («купил по ДДУ в ипотеку, а квартира с
  недоделками» → ДДУ + ипотека). Тема из меню бота — мягкая подсказка, **не** фильтр поиска.
* **Шаг 2 — поиск и генерация.** Векторный поиск по всей базе для каждого подзапроса, объединение
  уникальных фрагментов, генерация ответа с обязательными ссылками на статьи/законы/постановления.
* **Шаг 3 — проверка.** Отдельный вызов LLM сверяет черновик с найденным контекстом: нет ли
  ссылок на нормы, которых нет во фрагментах, и утверждений без опоры на источник. При проблемах —
  **один** повтор генерации (не бесконечный цикл).
* **История диалога.** Буфер «вопрос → ответ», подаётся на Шаг 1 и Шаг 2, чтобы обрабатывать
  уточнения («а если новый собственник поднимет плату?»).


## 4. Реализация алгоритма

Модель — `gpt-4o-mini`, `temperature=0` (для юридического ассистента нужна воспроизводимость и
минимум «фантазии»). Используется клиент `openai` напрямую.


In [ ]:
import json
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

def chat(system, user, temperature=0):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    return resp.choices[0].message.content

def parse_json_block(text):
    """Достаёт JSON из ответа модели (на случай ```json ... ``` обёртки)."""
    t = text.strip()
    if t.startswith("```"):
        t = t.strip("`")
        t = t[4:].strip() if t.lower().startswith("json") else t
    start, end = t.find("{"), t.rfind("}")
    return json.loads(t[start:end + 1])

### Шаг 1. Разбор вопроса и сборка поисковых запросов

In [ ]:
THEMES = ["купля-продажа", "наём/аренда", "ипотека", "ДДУ", "налоги",
          "регистрация", "дарение", "рента", "наследование", "общее"]

STEP1_SYSTEM = f"""Ты — ассистент юриста по сделкам физических лиц с ЖИЛОЙ недвижимостью.
Твоя задача — разобрать бытовой вопрос пользователя и подготовить запросы для поиска по базе
законов и судебной практики.

Сделай следующее:
1. Определи тему(ы) вопроса из списка: {THEMES}.
2. Если вопрос составной — раздели его на самостоятельные подвопросы.
3. Для каждого подвопроса сформулируй короткий юридически точный поисковый запрос
   (термины закона, а не бытовые слова).
4. Учитывай историю диалога: вопрос может быть уточнением к предыдущему.

Верни СТРОГО JSON без пояснений:
{{"themes": ["..."], "queries": ["...", "..."]}}"""

def step1_build_queries(question, history="", menu_theme=None):
    hint = f"\nПодсказка от пользователя (тема из меню): {menu_theme}" if menu_theme else ""
    user = f"История диалога:\n{history or '(пусто)'}{hint}\n\nВопрос пользователя:\n{question}"
    raw = chat(STEP1_SYSTEM, user)
    try:
        data = parse_json_block(raw)
        queries = [q for q in data.get("queries", []) if q.strip()]
        themes = data.get("themes", [])
    except Exception:
        queries, themes = [], []
    if not queries:                      # страховка, если модель не вернула JSON
        queries = [question]
    return themes, queries

### Шаг 2. Поиск в FAISS и генерация ответа

In [ ]:
def retrieve(queries, k=4):
    """Поиск по каждому запросу, объединение уникальных чанков."""
    seen, docs = set(), []
    for q in queries:
        for d in knowledge_base.similarity_search(q, k=k):
            key = (d.metadata["source"], d.page_content[:120])
            if key not in seen:
                seen.add(key)
                docs.append(d)
    return docs

def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        tag = CATEGORY_TITLES[d.metadata["category"]]
        parts.append(f"[Фрагмент {i}] источник: {d.metadata['source']} ({tag})\n{d.page_content}")
    return "\n\n".join(parts)

STEP2_SYSTEM = """Ты — нейроюрист. Консультируешь физических лиц ТОЛЬКО по сделкам с ЖИЛОЙ
недвижимостью (купля-продажа, наём, ипотека, ДДУ, дарение, рента, наследование жилья, налоги при
сделках с жильём, государственная регистрация прав).

Правила:
- Отвечай, опираясь ТОЛЬКО на приведённые фрагменты базы знаний и историю диалога.
- Каждый правовой тезис сопровождай ссылкой на конкретную норму из фрагментов
  (статья ГК/ЖК/СК/НК, номер ФЗ, номер постановления Пленума). Не выдумывай номера статей.
- Если во фрагментах недостаточно данных для ответа — прямо скажи об этом, не додумывай.
- Если вопрос не относится к сделкам с жилой недвижимостью — вежливо откажись консультировать.
- Пиши понятным языком для неюриста, по существу.
- В конце ответа перечисли использованные источники."""

def step2_generate(question, docs, history="", extra_instruction=""):
    context = format_context(docs)
    user = (f"Фрагменты базы знаний:\n{context}\n\n"
            f"История диалога:\n{history or '(пусто)'}\n\n"
            f"Вопрос пользователя:\n{question}")
    if extra_instruction:
        user += f"\n\nУчти замечания проверяющего и исправь ответ:\n{extra_instruction}"
    return chat(STEP2_SYSTEM, user)

### Шаг 3. Проверка ответа

In [ ]:
STEP3_SYSTEM = """Ты — проверяющий юрист-редактор. Тебе даны ЧЕРНОВИК ответа и ФРАГМЕНТЫ-источники,
на которых он должен быть основан.

Проверь:
1. Каждая ссылка на норму (статья, номер ФЗ, номер постановления Пленума) из черновика реально
   присутствует в фрагментах.
2. Нет утверждений о правовых последствиях, которые не подтверждаются фрагментами.
3. Ответ не выходит за тему жилой недвижимости.

Верни СТРОГО JSON без пояснений:
{"ok": true, "problems": []}
или
{"ok": false, "problems": ["конкретное замечание", "..."]}"""

def step3_verify(draft, docs):
    user = f"ФРАГМЕНТЫ-источники:\n{format_context(docs)}\n\nЧЕРНОВИК ответа:\n{draft}"
    raw = chat(STEP3_SYSTEM, user)
    try:
        data = parse_json_block(raw)
        return bool(data.get("ok")), data.get("problems", [])
    except Exception:
        return True, []          # если проверяющий не вернул JSON — не блокируем ответ

### Оркестратор: связываем три шага + история диалога

In [ ]:
def ask(question, history="", menu_theme=None, verbose=True):
    def log(*a):
        if verbose: print(*a)

    log("=" * 80)
    log("ВОПРОС:", question)
    if menu_theme:
        log("Тема из меню:", menu_theme)

    # --- Шаг 1
    themes, queries = step1_build_queries(question, history, menu_theme)
    log("\n[Шаг 1] Темы:", themes)
    log("[Шаг 1] Поисковые запросы:")
    for q in queries:
        log("   •", q)

    # --- Шаг 2
    docs = retrieve(queries)
    log(f"\n[Шаг 2] Найдено уникальных фрагментов: {len(docs)}")
    for d in docs:
        log("   -", d.metadata["source"])
    draft = step2_generate(question, docs, history)
    log("\n[Шаг 2] Черновик ответа:\n" + draft)

    # --- Шаг 3
    ok, problems = step3_verify(draft, docs)
    log("\n[Шаг 3] Проверка:", "OK" if ok else "ЕСТЬ ЗАМЕЧАНИЯ")
    answer = draft
    if not ok:
        for p in problems:
            log("   ! ", p)
        log("\n[Шаг 3] Повторная генерация с учётом замечаний...")
        answer = step2_generate(question, docs, history, extra_instruction="\n".join(problems))
        log("\n[Шаг 3] Исправленный ответ:\n" + answer)

    log("\n" + "-" * 80)
    log("ИТОГОВЫЙ ОТВЕТ:\n" + answer)
    log("=" * 80)
    return answer


class Dialog:
    """Тонкая обёртка для ведения истории диалога."""
    def __init__(self):
        self.history = ""
    def ask(self, question, menu_theme=None, verbose=True):
        answer = ask(question, self.history, menu_theme, verbose)
        self.history += f"Вопрос: {question}\nОтвет: {answer}\n\n"
        return answer

## 5. Демонстрация работы прототипа

Вопросы взяты из тестового набора `testset/test_questions.md` (сформулированы бытовым языком,
как реальные сообщения в мессенджере). Для каждого вопроса выводятся все промежуточные шаги
алгоритма и итоговый ответ.


### 5.1. Одиночные вопросы по разным темам

In [ ]:
demo_questions = [
    ("купля-продажа", "Подписали договор купли продажи. Квартира уже моя?"),
    ("купля-продажа", "У продавца в свидетельстве он один. Но он женат. Жену его спрашивать надо?"),
    ("наём/аренда",   "Снимаю квартиру. Хозяин ее продал. Мне съезжать?"),
    ("налоги",        "Продаю квартиру. Владею ей два года. Придется платить налог?"),
    ("наследование",  "Отец умер, осталась квартира. К нотариусу не ходил. Но живу там и плачу за коммуналку. Это считается, что я принял наследство?"),
]

for theme, q in demo_questions:
    ask(q, menu_theme=theme)
    print("\n\n")

### 5.2. Составной вопрос (две темы сразу)

Проверяем Шаг 1: вопрос должен разложиться на подвопросы по ДДУ и по ипотеке.


In [ ]:
ask("Купил квартиру по ДДУ в ипотеку. Дом сдали, а там куча недоделок. Что делать?",
    menu_theme="ДДУ")

### 5.3. Диалог с уточняющим вопросом

Второй вопрос не самодостаточен — он понятен только с учётом истории первого.


In [ ]:
d = Dialog()
d.ask("Снимаю квартиру по договору найма на год. Хозяин ее продал. Мне съезжать?",
      menu_theme="наём/аренда")
print("\n\n########## УТОЧНЯЮЩИЙ ВОПРОС ##########\n\n")
d.ask("А новый хозяин может поднять мне плату?")

### 5.4. Вопрос вне периметра

Прототип должен вежливо отказаться, а не консультировать по чужой теме.


In [ ]:
ask("Меня оштрафовали за парковку на газоне. Как обжаловать?")

## 6. Выводы

* **RAG-подход работает для юридической консультации.** Векторный поиск по базе из ~87 документов
  (законы + практика) находит релевантные нормы по бытовым формулировкам вопросов; генерация даёт
  ответ со ссылками на конкретные статьи.
* **Многошаговая цепочка повышает надёжность.** Отдельный шаг разбора вопроса (Шаг 1) корректно
  раскладывает составные вопросы на подвопросы и переводит бытовую речь в юридические термины,
  что заметно улучшает попадание поиска. Отдельный шаг проверки (Шаг 3) отлавливает ссылки на
  нормы, которых нет в найденном контексте, и запускает один повтор генерации.
* **`temperature=0`** для юридического ассистента оправдана: важнее воспроизводимость и опора на
  источник, чем разнообразие формулировок.
* **Ограничение периметра** (только жилая недвижимость) удерживается на уровне системных промптов
  Шага 2 и Шага 3 — модель отказывается отвечать на вопросы вне темы.
* **Слабые места прототипа:** качество ответа чувствительно к размеру чанка и числу извлекаемых
  фрагментов `k`; при «пограничных» вопросах поиск иногда приносит смежные, но не самые точные
  статьи; проверка на Шаге 3 не гарантирует юридической полноты ответа, только его соответствие
  найденным фрагментам.

## 7. План дальнейшей работы

1. **Экспериментальная часть (Этап № 4).** Таблица сравнения гиперпараметров на тестовом наборе из
   30 вопросов: `chunk_size` / `chunk_overlap`, число фрагментов `k`, модель эмбеддингов,
   версии промптов; метрика — доля вопросов, где найден эталонный источник и ответ на него корректно
   ссылается. Зафиксировать итоговую конфигурацию с обоснованием.
2. **Метаданные чанков.** Проставлять при индексации источник, дату проверки и статус редакции
   документа (сейчас ведётся вручную в `MANIFEST.md`) — и показывать пользователю дату актуальности базы.
3. **Продакшн-интеграция.** Вынести пайплайн в FastAPI-сервис с эндпоинтом `/ask` (классификация,
   поиск, генерация, проверка, история — на бэкенде) и подключить Telegram-бот как тонкий клиент:
   меню тем, текстовый и голосовой ввод (`whisper-1`).
4. **Регламент актуализации базы.** Ежеквартальная сверка редакций законов и новых обзоров практики
   ВС РФ + служебный скрипт полной пересборки индекса FAISS при обновлении файлов базы.
5. **Расширение оценки качества.** Помимо «найден ли эталонный источник» — экспертная проверка
   юридической корректности ответов на репрезентативной выборке.
